# Part 1: Neural Network Fundamentals and Training Behavior Analysis
## Customer Churn Prediction — Q1 Dataset
**Course:** Applied Neural Networks | **Student Submission**

---

In this notebook, I'm going to build a neural network from scratch (well, using scikit-learn's `MLPClassifier`) to predict whether a customer will churn. I'll walk through every step from loading the data all the way to a hyperparameter comparison table. Let's go!


## Setup — Importing Libraries

In [ ]:
# all the libraries we'll need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                             confusion_matrix, classification_report,
                             roc_auc_score)
from sklearn.utils.class_weight import compute_sample_weight

print("All libraries imported successfully!")


---
## Task 1: Dataset Understanding

First things first — let's load the data and see what we're working with.


In [ ]:
# loading the dataset
df = pd.read_csv('Q1_customer_churn_nn.csv')

print("Shape of dataset:", df.shape)
print()
print("First 5 rows:")
df.head()


In [ ]:
# checking column names and data types
print("Column names and data types:")
print(df.dtypes)


In [ ]:
# checking for missing values — always important to do this early
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Total missing values:", df.isnull().sum().sum())


In [ ]:
# basic statistical summary of numerical columns
df.describe()


In [ ]:
# checking how the target variable is distributed
print("Target variable (churn) distribution:")
print(df['churn'].value_counts())
print()
print("As percentages:")
print(df['churn'].value_counts(normalize=True).round(4) * 100, "%")


In [ ]:
# visualizing the class imbalance
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['churn'].value_counts()
ax.bar(['Retained (0)', 'Churned (1)'], counts.values,
       color=['steelblue', 'tomato'], edgecolor='black')
for i, v in enumerate(counts.values):
    ax.text(i, v + 10, str(v), ha='center', fontweight='bold', fontsize=12)
ax.set_title('Churn Distribution (Target Variable)', fontsize=13)
ax.set_ylabel('Number of Customers')
plt.tight_layout()
plt.savefig('results/churn_distribution.png', dpi=150)
plt.show()
print("Class imbalance is very noticeable here — 1969 retained vs only 31 churned!")
print("We'll need to handle this during training.")


### Task 1 Summary

| Property | Value |
|---|---|
| Rows | 2000 |
| Columns | 17 |
| Target Column | `churn` (0 = retained, 1 = churned) |
| Missing Values | None |
| Categorical Features | `region`, `plan_type`, `contract_type`, `payment_method` |
| Numerical Features | `tenure_months`, `monthly_charges_inr`, `avg_login_days_per_month`, `support_tickets_last_90_days`, `payment_delay_days`, `data_usage_gb`, `satisfaction_score`, `last_complaint_days_ago`, `discount_percent`, `autopay_enabled`, `referral_count` |
| Identifier (excluded) | `customer_id` |

One big thing I noticed right away: the dataset is **heavily imbalanced** — only 1.55% of customers actually churned. This is a real problem for training, and I'll address it using class weights.


---
## Task 2: Data Preprocessing

Now let's get the data ready for the neural network. The main things to do are:
1. Drop the customer ID (it's just an identifier, not a real feature)
2. Encode the categorical columns into numbers
3. Scale all the numerical features so they're on a similar range
4. Split into train and test sets


In [ ]:
# step 1 — drop the customer_id column, it's not useful as a feature
df_clean = df.drop(columns=['customer_id'])
print("Columns after dropping customer_id:", df_clean.shape[1])


In [ ]:
# step 2 — encoding categorical columns using LabelEncoder
# we're using label encoding since the categories don't have a natural order hierarchy
# (one-hot would also work but adds a lot more columns)

cat_cols = ['region', 'plan_type', 'contract_type', 'payment_method']
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])
    label_encoders[col] = le
    print(f"{col}: {list(le.classes_)} -> {list(range(len(le.classes_)))}")


In [ ]:
# step 3 — separating features (X) and target (y)
X = df_clean.drop(columns=['churn'])
y = df_clean['churn']

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)


In [ ]:
# step 4 — train/test split (80% train, 20% test)
# using stratify=y to make sure both splits have the same churn ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)
print()
print("Churn rate in training set:", round(y_train.mean() * 100, 2), "%")
print("Churn rate in test set:", round(y_test.mean() * 100, 2), "%")


In [ ]:
# step 5 — scaling the numerical features using StandardScaler
# this is important for neural networks because they're sensitive to feature magnitudes
# the scaler is fitted ONLY on training data to avoid data leakage

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # just transform, don't fit again!

print("Sample of scaled training features (first row):")
print(np.round(X_train_scaled[0], 3))


In [ ]:
# handling class imbalance using sample weights
# instead of the model just learning to always predict "retained",
# we tell it to pay more attention to the rare churn cases during training

sample_weights = compute_sample_weight('balanced', y_train)
print("Sample weight for retained customer:", round(sample_weights[y_train.values[0]], 4))
print("Sample weight for churned customer:", round(sample_weights[y_train[y_train==1].index[0] - y_train.index[0]], 4))
print()
print("This makes the model treat each churned customer as roughly 63x more important during training!")


---
## Task 3: Neural Network Model Building

For this task I'm using `scikit-learn`'s `MLPClassifier`, which is a proper feed-forward neural network. The name stands for **Multi-Layer Perceptron**, which is exactly the kind of network we've been learning about.

Here's my baseline architecture:
- **Input layer**: 15 neurons (one per feature)
- **Hidden layer 1**: 64 neurons with ReLU activation
- **Hidden layer 2**: 32 neurons with ReLU activation  
- **Output layer**: 1 neuron with sigmoid (logistic) activation → outputs a probability for churn

For the loss function, it uses **binary cross-entropy** under the hood (since `churn` is 0 or 1), and the optimizer is **Adam** (`solver='adam'` in sklearn).


In [ ]:
# building the baseline neural network
# architecture: Input(15) -> Dense(64, ReLU) -> Dense(32, ReLU) -> Output(1, Sigmoid)

model = MLPClassifier(
    hidden_layer_sizes=(64, 32),   # two hidden layers: 64 then 32 neurons
    activation='relu',             # ReLU activation in hidden layers
    solver='adam',                 # Adam optimizer
    learning_rate_init=0.001,      # starting learning rate
    batch_size=32,                 # mini-batch size
    max_iter=200,                  # number of training epochs
    random_state=42,               # for reproducibility
    verbose=False
)

print("Model architecture:")
print("  Input layer:    15 features (neurons)")
print("  Hidden layer 1: 64 neurons — ReLU activation")
print("  Hidden layer 2: 32 neurons — ReLU activation")
print("  Output layer:   1 neuron — Sigmoid (logistic) activation")
print("  Loss function:  Binary Cross-Entropy")
print("  Optimizer:      Adam (lr=0.001)")
print("  Batch size:     32")
print("  Max epochs:     200")


---
## Task 4: Training and Evaluation

Let's train the model and see how it does!


In [ ]:
# training the model — passing in sample_weights to handle class imbalance
model.fit(X_train_scaled, y_train, sample_weight=sample_weights)
print("Training complete!")
print(f"Number of iterations run: {model.n_iter_}")


In [ ]:
# checking train vs test accuracy
train_preds = model.predict(X_train_scaled)
test_preds  = model.predict(X_test_scaled)

train_acc = accuracy_score(y_train, train_preds)
test_acc  = accuracy_score(y_test, test_preds)

print(f"Training Accuracy: {train_acc:.4f}  ({train_acc*100:.2f}%)")
print(f"Test Accuracy:     {test_acc:.4f}  ({test_acc*100:.2f}%)")


In [ ]:
# plotting the training loss curve
plt.figure(figsize=(8, 4))
plt.plot(model.loss_curve_, color='steelblue', linewidth=2)
plt.title('Training Loss Curve (Baseline Model)', fontsize=13)
plt.xlabel('Epoch')
plt.ylabel('Loss (Binary Cross-Entropy)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/loss_curve.png', dpi=150)
plt.show()
print("The loss is going down nicely — the model is learning!")


In [ ]:
# confusion matrix — this is more informative than raw accuracy for imbalanced data
cm = confusion_matrix(y_test, test_preds)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Retained', 'Churned'],
            yticklabels=['Retained', 'Churned'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix – Baseline Model', fontsize=13)
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', dpi=150)
plt.show()


In [ ]:
# full classification report
print("Classification Report:")
print(classification_report(y_test, test_preds,
      target_names=['Retained (0)', 'Churned (1)'], zero_division=0))

# AUC-ROC score
test_proba = model.predict_proba(X_test_scaled)[:, 1]
auc = roc_auc_score(y_test, test_proba)
print(f"AUC-ROC Score: {auc:.4f}")


### Interpretation

Accuracy is 97.5%, which sounds impressive but is a bit misleading here because the dataset is so imbalanced — if the model just predicted "retained" for everyone it would get ~98.4% accuracy without learning anything useful!

The **AUC-ROC of 0.71** and the **F1-score for the churn class** are better indicators. The confusion matrix shows the model is catching some churners but still missing quite a few — this is a hard problem with only 31 churn cases in 2000 rows.

The good news is the training loss curve is smooth and decreasing, which means the model is actually learning the patterns in the data.


---
## Task 5: Hyperparameter Experimentation

I ran 4 different configurations to see how changing the architecture and hyperparameters affects performance. The things I changed include: number of layers/neurons, learning rate, batch size, and activation function.


In [ ]:
# defining all 4 experiment configurations
experiments = [
    {
        'name': 'Exp 1 – Baseline',
        'hidden_layer_sizes': (64, 32),
        'activation': 'relu',
        'learning_rate_init': 0.001,
        'batch_size': 32,
        'max_iter': 200,
        'description': '2 hidden layers (64→32), ReLU, lr=0.001, batch=32'
    },
    {
        'name': 'Exp 2 – Deeper + tanh',
        'hidden_layer_sizes': (128, 64, 32),
        'activation': 'tanh',
        'learning_rate_init': 0.001,
        'batch_size': 32,
        'max_iter': 200,
        'description': '3 hidden layers (128→64→32), tanh, lr=0.001, batch=32'
    },
    {
        'name': 'Exp 3 – High Learning Rate',
        'hidden_layer_sizes': (64, 32),
        'activation': 'relu',
        'learning_rate_init': 0.01,
        'batch_size': 64,
        'max_iter': 200,
        'description': '2 hidden layers (64→32), ReLU, lr=0.01, batch=64'
    },
    {
        'name': 'Exp 4 – Smaller Network + Sigmoid',
        'hidden_layer_sizes': (32,),
        'activation': 'logistic',
        'learning_rate_init': 0.0005,
        'batch_size': 16,
        'max_iter': 200,
        'description': '1 hidden layer (32), Sigmoid, lr=0.0005, batch=16'
    },
]


In [ ]:
# running all 4 experiments and collecting results
results_list = []
loss_curves_all = []

for exp in experiments:
    name = exp['name']
    desc = exp['description']
    
    # build and train the model
    m = MLPClassifier(
        hidden_layer_sizes=exp['hidden_layer_sizes'],
        activation=exp['activation'],
        solver='adam',
        learning_rate_init=exp['learning_rate_init'],
        batch_size=exp['batch_size'],
        max_iter=exp['max_iter'],
        random_state=42
    )
    m.fit(X_train_scaled, y_train, sample_weight=sample_weights)
    
    # get predictions
    tr_preds = m.predict(X_train_scaled)
    te_preds = m.predict(X_test_scaled)
    te_proba = m.predict_proba(X_test_scaled)[:, 1]
    
    # calculate metrics
    tr_acc  = accuracy_score(y_train, tr_preds)
    te_acc  = accuracy_score(y_test, te_preds)
    f1      = f1_score(y_test, te_preds, zero_division=0)
    auc_val = roc_auc_score(y_test, te_proba)
    
    results_list.append({
        'Experiment': name,
        'Configuration': desc,
        'Train Acc': round(tr_acc, 4),
        'Test Acc': round(te_acc, 4),
        'F1-Score (Churn)': round(f1, 4),
        'AUC-ROC': round(auc_val, 4)
    })
    loss_curves_all.append((name, m.loss_curve_))
    
    print(f"{name}: test_acc={te_acc:.4f}  f1={f1:.4f}  auc={auc_val:.4f}")

print("\nAll experiments done!")


In [ ]:
# printing the comparison table
results_df = pd.DataFrame(results_list)
print("\n=== Hyperparameter Comparison Table ===\n")
print(results_df.to_string(index=False))
results_df.to_csv('results/model_comparison_table.csv', index=False)


In [ ]:
# saving the comparison table as a nice image
fig, ax = plt.subplots(figsize=(14, 2.6))
ax.axis('off')

display_df = results_df.drop(columns=['Configuration'])
tbl = ax.table(cellText=display_df.values, colLabels=display_df.columns,
               cellLoc='center', loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 2.4)

# coloring the header row
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2c7bb6')
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#dceefb')

ax.set_title('Hyperparameter Experiment Comparison Table', fontsize=13,
             pad=10, fontweight='bold')
plt.tight_layout()
plt.savefig('results/model_comparison_table.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# plotting loss curves for all experiments on one chart
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue', 'tomato', 'forestgreen', 'darkorange']

for (name, lc), col in zip(loss_curves_all, colors):
    ax.plot(lc, label=name, color=col, linewidth=2)

ax.set_title('Loss Curves — All Hyperparameter Experiments', fontsize=13)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/all_loss_curves.png', dpi=150)
plt.show()


In [ ]:
# saving the combined evaluation outputs panel
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# panel 1 — churn distribution
counts = df['churn'].value_counts()
axes[0].bar(['Retained (0)', 'Churned (1)'], counts.values,
            color=['steelblue', 'tomato'], edgecolor='black')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')
axes[0].set_title('Churn Distribution')
axes[0].set_ylabel('Count')

# panel 2 — confusion matrix (best experiment = exp 4)
_, best_lc = loss_curves_all[3]
best_m = MLPClassifier(hidden_layer_sizes=(32,), activation='logistic',
                       solver='adam', learning_rate_init=0.0005,
                       batch_size=16, max_iter=200, random_state=42)
best_m.fit(X_train_scaled, y_train, sample_weight=sample_weights)
best_preds = best_m.predict(X_test_scaled)
cm_best = confusion_matrix(y_test, best_preds)
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Retained', 'Churned'],
            yticklabels=['Retained', 'Churned'], ax=axes[1])
axes[1].set_title('Confusion Matrix (Exp 4 — Best AUC)')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')

# panel 3 — all loss curves
for (name, lc), col in zip(loss_curves_all, colors):
    axes[2].plot(lc, label=name.split('–')[0].strip(), color=col, linewidth=1.8)
axes[2].set_title('All Loss Curves')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Loss')
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

plt.suptitle('Evaluation Outputs — Customer Churn Neural Network', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/evaluation_outputs.png', dpi=150, bbox_inches='tight')
plt.show()
print("All evaluation output figures saved!")


### Experiment Analysis

Here's what I found from the 4 experiments:

- **Exp 1 (Baseline)** performed well on raw accuracy but had low F1 on the churn class (0.17). Train accuracy hit 100% which is a sign of overfitting.
- **Exp 2 (Deeper + tanh)**: Adding more layers and switching to tanh improved the AUC slightly but didn't help with F1. The model still struggled to identify churners.
- **Exp 3 (Higher LR)**: Increasing the learning rate to 0.01 made the loss curve noisier and didn't improve performance much. This showed me firsthand how sensitive the learning rate is — even a 10x increase hurt consistency.
- **Exp 4 (Smaller + Sigmoid + low LR)**: This was the most interesting result! The simpler model with a single 32-neuron layer and a very small learning rate of 0.0005 got an **AUC-ROC of 0.967** and the best F1-score for churn (0.36). I think the smaller network was forced to generalize better instead of memorizing the training data, and the slow learning rate let it settle more carefully. The gap between train (94.5%) and test (96.5%) accuracy is also the smallest, suggesting less overfitting.


---
## Task 6: Final Reflection

### What role do weights and biases play in the model?

Weights and biases are literally what the neural network learns. Every connection between neurons has a weight — it controls how much influence that input has on the next neuron. The bias is a constant added to the weighted sum, which lets the neuron "shift" its activation threshold. Without biases, every neuron would always output zero when all inputs are zero, which limits what the model can learn. During training, the backpropagation algorithm adjusts all these weights and biases step by step to reduce the loss. By the end of training, the weights basically encode all the patterns the model picked up from the data.

### Why is an activation function required?

Without an activation function, stacking multiple layers would be pointless — mathematically, a stack of linear transformations is just one big linear transformation. Activation functions like ReLU introduce non-linearity, which is what allows neural networks to learn complex, curved decision boundaries. In our churn dataset, the relationship between features like `satisfaction_score` or `payment_delay_days` and churn isn't a straight line — the activation function is what lets the model capture those non-linear patterns.

### What happens when the learning rate is too high or too low?

When I ran Exp 3 with `lr=0.01` (10x higher than baseline), the loss curve was noisier and the model didn't converge as cleanly. A learning rate that's too high causes the parameter updates to overshoot the minimum — the model keeps bouncing around instead of settling. On the other end, a learning rate that's too low (like an extreme `lr=0.00001`) would mean the model takes forever to learn, and might get stuck in a local minimum before reaching a good solution. We found that `lr=0.001` and `lr=0.0005` both worked well here.

### Did the model show signs of underfitting or overfitting?

The baseline model (Exp 1 and Exp 2) showed **clear signs of overfitting** — training accuracy was 99.9-100% while test accuracy was lower and F1 on the minority class was weak. The model basically memorized the training data, especially the majority class.

Exp 4 showed the least gap between train and test accuracy (~94.5% vs 96.5%), which is actually a healthier sign — though the reason test accuracy is slightly *higher* than train accuracy here is likely because the balanced sample weights make training harder, while at test time we predict without any weighting. Overall, the imbalanced dataset is the biggest challenge — no config achieved a really high F1 for the churn class without trading off something else.


---
## Requirements

See `requirements.txt` for the full list of dependencies.

In [ ]:
# printing the library versions used
import sklearn, numpy, pandas, matplotlib, seaborn
print(f"scikit-learn: {sklearn.__version__}")
print(f"numpy:        {numpy.__version__}")
print(f"pandas:       {pandas.__version__}")
print(f"matplotlib:   {matplotlib.__version__}")
print(f"seaborn:      {seaborn.__version__}")
